In [ ]:
#imports
import numpy as np
import pandas as pd
import os


In [ ]:
#open folder with openface files
folder = 'OpenFace_2.2.0_win_x64/processed/'

folder_list = os.listdir(folder)
folder_list = [f for f in folder_list if 'all' not in f]
folder_list = [f for f in folder_list if 'OLD' not in f]

#save results in df
df_facequal = pd.DataFrame()

for fold in folder_list:
    #get csv files
    files = os.listdir(folder + fold)
    files = [f for f in files if f.endswith('.csv')]
    for file in files:
        #read csv file
        df = pd.read_csv(folder + fold + '/' + file)
        #participant number is file name
        participant = file.split('.')[0]
        #get confidence scores and see percentage above 0.5 and o.7
        total_frames = df.shape[0]
        df_05 = df[df[' confidence'] > 0.5]
        above_05 = df_05.shape[0]/total_frames
        df_07 = df[df[' confidence'] > 0.7]
        above_07 = df_07.shape[0]/total_frames
        #average confidence
        avg_confidence = df[' confidence'].mean()

        if above_07 < 0.5:
            print('Low confidence:', participant,above_07)

        #see how many frames may be missing
        frames = df['frame']
        missing_frames = 0
        for i in range(1, frames.iloc[-1]):
            if i not in frames.values:
                missing_frames += 1
        #print('Missing frames:', missing_frames)
        missing_frames_per = missing_frames / frames.iloc[-1]
        missing_frames_total = missing_frames

        #save results in df
        df_facequal_part = pd.DataFrame({'participant': participant, 'total_frames': total_frames, 'above_05': above_05, 'above_07': above_07, 'average_confidence': avg_confidence,'missing_frames': missing_frames_total,'missing_frames_percentage':missing_frames_per}, index=[0])
        df_facequal = pd.concat([df_facequal, df_facequal_part], ignore_index=True)

#save df
df_facequal.to_csv('openface_predictions.csv', index=False)

#print average and std of confidence, for 0.5 and 0.7
print('Average confidence above 0.5:', df_facequal['above_05'].sum() / df_facequal['total_frames'].sum())
print('Std confidence above 0.5:', df_facequal['above_05'].std())
print('Average confidence above 0.7:', df_facequal['above_07'].sum() / df_facequal['total_frames'].sum())
print('Std confidence above 0.7:', df_facequal['above_07'].std())

print('Average confidence:', df_facequal['average_confidence'].mean())
print('Max confidence:', df_facequal['average_confidence'].max())
#session with max confidence
print('Max confidence session:', df_facequal[df_facequal['average_confidence'] == df_facequal['average_confidence'].max()])
print('Min confidence:', df_facequal['average_confidence'].min())
#session with min confidence
print('Min confidence session:', df_facequal[df_facequal['average_confidence'] == df_facequal['average_confidence'].min()])

